<a href="https://colab.research.google.com/github/mjjaiavinash/avinash-codebooster-2026/blob/main/Day%204/day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install spark --quiet
print("Pyspark Installed Successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.7/351.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.8 MB/s eta 0:00:00
Pyspark Installed Successfully


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month, to_date, col, round as spark_round
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [4]:
spark = SparkSession.builder\
.appName('Day4_BigData_Sales')\
.config('spark.sql.adaptive.enabled','true')\
.getOrCreate()

print("Spark Version:",spark.version)
print("SparkSession:","ACTIVE")
print("Application:",spark.sparkContext.appName)


Spark Version: 4.0.2
SparkSession: ACTIVE
Application: Day4_BigData_Sales


In [5]:
df_bronze = spark.read\
.option('header','true')\
.option('inferSchema','true')\
.csv('/content/drive/MyDrive/15 days intern dataset/large_sales_data.csv')

In [6]:
print("==== Bronze Layer Raw Data ====")
print()
print("No of Rows:",df_bronze.count())
print("No of Columns:", len(df_bronze.columns))

==== Bronze Layer Raw Data ====

No of Rows: 5000
No of Columns: 13


In [7]:
print("===== First Five Rows =====")
print()
df_bronze.show(5, truncate=False)

print()

print("===== Last Five Rows =====")
print()
last = df_bronze.tail(5)
spark.createDataFrame(last, df_bronze.schema).show(truncate=False)

===== First Five Rows =====

+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    

In [8]:
df_bronze.write\
.mode('OverWrite')\
.parquet('sales_bronze.parquet')

In [9]:
import os

def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path)/1024
  total = 0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath,f))
  return total/1024

csv_size = get_dir_size('/content/drive/MyDrive/15 days intern dataset/large_sales_data.csv')
parquet_size = get_dir_size('/content/sales_bronze.parquet')
reduction = (1-parquet_size/csv_size)*100


print("CSV Size:", csv_size, "KB")
print("Parquet Size:", parquet_size, "KB")
print(f"Reduction in Size: {reduction:.1f} %")

CSV Size: 529.3125 KB
Parquet Size: 55.09765625 KB
Reduction in Size: 89.6 %


In [10]:
df_bronze = df_bronze.withColumn(
    "total_price",
    col("unit_price") + col("revenue")
)

print("===== Unit Price + Revenue =====")
df_bronze.select(
    "unit_price",
    "revenue",
    "total_price"
).show()

===== Unit Price + Revenue =====
+----------+-------+-----------+
|unit_price|revenue|total_price|
+----------+-------+-----------+
|     22000| 264000|     286000|
|     12000| 120000|     132000|
|       800|   8000|       8800|
|     32000| 160000|     192000|
|      3500|  14000|      17500|
|      2500|  25000|      27500|
|       600|   5400|       6000|
|     45000| 585000|     630000|
|      3500|  38500|      42000|
|      3500|  35000|      38500|
|       600|   9000|       9600|
|       800|  10400|      11200|
|      4500|  18000|      22500|
|      3500|  14000|      17500|
|      2500|   7500|      10000|
|      1200|  15600|      16800|
|      4500|  40500|      45000|
|      2500|  12500|      15000|
|     12000| 108000|     120000|
|      2500|  22500|      25000|
+----------+-------+-----------+
only showing top 20 rows


In [28]:
df_silver=df_bronze \
    .dropDuplicates() \
    .dropna(subset=['quantity','unit_price','revenue'])

df_silver=df_silver.withColumn("order_date",to_date(col("order_date"),"yyyy-MM-dd")) \
    .withColumn("year",year(col("order_date"))) \
    .withColumn("month",month(col("order_date")))

df_sliver=df_sliver.withColumn("revenue_category",F.when(col('revenue')>40000,'High')\
                                        .when((col('revenue')>10000) & (col('revenue')<=40000),'Medium')\
                                        .otherwise('Low'))
df_silver.show(5)
print(f'Silver table row count: {df_silver.count()}')
print(f'Silver table column count: {len(df_silver.columns)}')

+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+-----------+----+-----+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|  sales_rep|  payment_method|order_status|total_price|year|month|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+-----------+----+-----+
|    1121|   Suresh Rao|Monitor|Electronics|      11|     22000| 242000|2023-11-08|   Mumbai|  West|Meera Patel|             UPI|  Processing|     264000|2023|   11|
|    1842|  Priya Patel| Webcam|Accessories|      11|      2500|  27500|2023-01-22|Ahmedabad|  West| Priya Nair|Cash on Delivery|  Processing|      30000|2023|    1|
|    2296|  Divya Singh|Printer|Electronics|      13|     12000| 156000|2023-09-29|Ahmedabad|  West| Sunita Rao|     Credit Card|     Shipped|     168000|2023|    9|
|   

In [29]:
df_silver.write\
.mode('overwrite')\
.parquet('sales_silver.parquet')

print('silver parquet saved: sales_silver.parquet')


silver parquet saved: sales_silver.parquet


In [30]:
top_products = (
    df_silver
    .groupBy('product')
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.count('order_id').alias('num_orders'),
        F.avg('revenue').alias('avg_order_revenue')
    )\
    .orderBy('total_revenue',ascending = False)
    .limit(5)
)

print("==== Top 5 products by revenue ====")
top_products.show(truncate = False)

==== Top 5 products by revenue ====
+-------+-------------+----------+------------------+
|product|total_revenue|num_orders|avg_order_revenue |
+-------+-------------+----------+------------------+
|Laptop |182700000    |502       |363944.22310756973|
|Tablet |135104000    |532       |253954.8872180451 |
|Monitor|82126000     |481       |170740.12474012474|
|Printer|44544000     |488       |91278.68852459016 |
|Speaker|16317000     |470       |34717.02127659575 |
+-------+-------------+----------+------------------+



In [31]:
print("==== Top 10 Products by revenue Descending ====")
top_products_desc = (
    df_silver
    .groupBy("product")
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.count("order_id").alias("num_orders"),
        F.avg("revenue").alias("avg_order_revenue")
    )
    .orderBy(F.desc("total_revenue"))
    .limit(10)
)

top_products_desc.show(truncate=False)

print("==== Top 10 Products by revenue Ascending ====")
top_products_asc = (
    df_silver
    .groupBy("product")
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.count("order_id").alias("num_orders"),
        F.avg("revenue").alias("avg_order_revenue")
    )
    .orderBy(F.asc("total_revenue"))
    .limit(10)
)

top_products_asc.show(truncate=False)

==== Top 10 Products by revenue Descending ====
+----------+-------------+----------+------------------+
|product   |total_revenue|num_orders|avg_order_revenue |
+----------+-------------+----------+------------------+
|Laptop    |182700000    |502       |363944.22310756973|
|Tablet    |135104000    |532       |253954.8872180451 |
|Monitor   |82126000     |481       |170740.12474012474|
|Printer   |44544000     |488       |91278.68852459016 |
|Speaker   |16317000     |470       |34717.02127659575 |
|Headphones|13541500     |481       |28152.806652806652|
|Webcam    |10982500     |532       |20643.796992481202|
|Keyboard  |4878000      |495       |9854.545454545454 |
|Mouse     |3207200      |492       |6518.69918699187  |
|USB Hub   |2447400      |527       |4644.022770398482 |
+----------+-------------+----------+------------------+

==== Top 10 Products by revenue Ascending ====
+----------+-------------+----------+------------------+
|product   |total_revenue|num_orders|avg_order_re

In [32]:
revenue_by_region = (
    df_silver
    .groupBy("region")
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.count("order_id").alias("total_orders"),
        F.countDistinct("customer_name").alias("unique_customers")
    )
    .orderBy(F.desc("total_revenue"))
)

print("===== Revenue by Region =====")
revenue_by_region.show(truncate=False)

===== Revenue by Region =====
+------+-------------+------------+----------------+
|region|total_revenue|total_orders|unique_customers|
+------+-------------+------------+----------------+
|West  |198275600    |2021        |15              |
|South |147145900    |1483        |15              |
|North |99878400     |995         |15              |
|East  |50547700     |501         |15              |
+------+-------------+------------+----------------+



In [36]:
from pyspark.sql.functions import month

monthly_revenue_trend = (
    df_silver
    .withColumn("order_month", month("order_date"))
    .groupBy("order_month")
    .agg(
        F.sum("revenue").alias("monthly_revenue"),
        F.count("order_id").alias("monthly_orders")
    )
    .orderBy("order_month")
)

print("===== Monthly Revenue Trend =====")
monthly_revenue_trend.show(truncate=False)

===== Monthly Revenue Trend =====
+-----------+---------------+--------------+
|order_month|monthly_revenue|monthly_orders|
+-----------+---------------+--------------+
|1          |41068200       |423           |
|2          |34485400       |375           |
|3          |40031200       |451           |
|4          |38857100       |390           |
|5          |39984500       |423           |
|6          |40707400       |390           |
|7          |42640700       |405           |
|8          |43718500       |418           |
|9          |37640200       |398           |
|10         |47839000       |479           |
|11         |44577100       |419           |
|12         |44298300       |429           |
+-----------+---------------+--------------+



In [38]:
from pyspark.sql.functions import month, date_format

monthly_revenue_trend = (
    df_silver
    .withColumn("month_num", month("order_date"))
    .withColumn("order_month", date_format("order_date", "MMMM"))
    .groupBy("month_num", "order_month")
    .agg(
        F.sum("revenue").alias("monthly_revenue"),
        F.count("order_id").alias("monthly_orders")
    )
    .orderBy("month_num")
    .drop("month_num")
)

monthly_revenue_trend.show(truncate=False)

+-----------+---------------+--------------+
|order_month|monthly_revenue|monthly_orders|
+-----------+---------------+--------------+
|January    |41068200       |423           |
|February   |34485400       |375           |
|March      |40031200       |451           |
|April      |38857100       |390           |
|May        |39984500       |423           |
|June       |40707400       |390           |
|July       |42640700       |405           |
|August     |43718500       |418           |
|September  |37640200       |398           |
|October    |47839000       |479           |
|November   |44577100       |419           |
|December   |44298300       |429           |
+-----------+---------------+--------------+

